In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mplsoccer import Pitch


# 1. CARGA Y UNIFICACIÓN DE DATASETS


# Cargamos el Primer Tiempo (PT) y Segundo Tiempo (ST) limpios
df_pt = pd.read_csv('events-pt-limpio.csv')
df_st = pd.read_csv('events-st-limpio.csv')

# Añadimos la etiqueta de tiempo para mantener la trazabilidad
df_pt['tiempo'] = '1T'
df_st['tiempo'] = '2T'

# Consolidamos el dataset completo del partido
df = pd.concat([df_pt, df_st], ignore_index=True)

# Normalizamos cadenas de texto para evitar errores de tipeo
df['Player'] = df['Player'].str.strip().str.lower()
df['Event'] = df['Event'].str.strip().str.lower()

print(f"Dataset de 8va consolidado con éxito. Total de eventos: {len(df)}")



# 2. GRÁFICO 1: RIESGO EN SALIDA

# Filtramos pérdidas y despejes en campo propio (X < 60)
df_salida = df[
    (df['Event'].isin(['perdida', 'despeje'])) & 
    (df['X'] < 60)
]

# Inicializamos la cancha de mplsoccer (ajustá las dimensiones si usás StatsBomb o 0-100)
pitch = Pitch(pitch_type='custom', pitch_length=100, pitch_width=100, 
              pitch_color='#aabb99', line_color='white', stripe=True)

fig, ax = pitch.draw(figsize=(10, 7))

# Graficamos los eventos de riesgo en salida
for _, row in df_salida.iterrows():
    color = 'red' if row['Event'] == 'perdida' else 'yellow'
    marker = 'X' if row['Event'] == 'perdida' else 'o'
    
    pitch.scatter(row['X'], row['Y'], ax=ax, color=color, 
                  edgecolors='black', marker=marker, s=150, zorder=3)

ax.set_title('Riesgo en Salida - Eventos en Campo Propio (8va)', fontsize=16, pad=15)
plt.savefig('riesgo en salida_RECONSTRUIDO.png', bbox_inches='tight')
plt.show()



# 3. GRÁFICO 2: RENDIMIENTO EN DUELOS Y ANTICIPOS

# Filtramos los eventos de disputa defensiva
eventos_duelos = ['duelo_ganado', 'anticipo', 'duelo_perdido']
df_duelos = df[df['Event'].isin(eventos_duelos)]

plt.figure(figsize=(12, 6))
# Usamos el orden alfabético de los apellidos que tenías (diprofio, eberle, epeloa, etc.)
sns.countplot(
    data=df_duelos, 
    x='Player', 
    hue='Event', 
    palette={'duelo_ganado': '#2ecc71', 'anticipo': '#3498db', 'duelo_perdido': '#e74c3c'},
    order=sorted(df_duelos['Player'].unique())
)

plt.title('Rendimiento en Duelos y Anticipos por Jugador', fontsize=14, pad=15)
plt.xlabel('Jugador')
plt.ylabel('Cantidad de Eventos')
plt.xticks(rotation=45)
plt.legend(title='Tipo de Evento')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig('rendimiento en duelos y anticipos por jugador_RECONSTRUIDO.png', bbox_inches='tight')
plt.show()



# 4. GRÁFICO 3: PRECISIÓN VS RECUPERACIÓN

# Para este gráfico calculamos las métricas agregadas por cada futbolista
jugadores = df['Player'].unique()
resumen_jugadores = []

for j in jugadores:
    df_j = df[df['Player'] == j]
    
    # Métricas de pases (asumiendo que registrás 'pase_corto', 'pase_largo', 'centro', etc.)
    pases_totales = df_j[df_j['Event'].str.contains('pase|centro', na=False)]
    # Si tenés columna de efectividad o discriminás pérdidas directas de pases:
    perdidas = len(df_j[df_j['Event'] == 'perdida'])
    
    # Recuperaciones y Anticipos
    recuperaciones = len(df_j[df_j['Event'].isin(['recuperacion', 'anticipo', 'duelo_ganado'])])
    
    # Un cálculo estimado de precisión basado en volumen de juego (ajustalo a tus columnas exactas si usás etiquetas de acierto/error)
    total_acciones = len(df_j)
    precision_estimada = ((total_acciones - perdidas) / total_acciones) * 100 if total_acciones > 0 else 0
    
    if total_acciones > 2: # Filtramos jugadores con muy poca participación
        resumen_jugadores.append({
            'Jugador': j,
            'Precision': precision_estimada,
            'Recuperaciones': recuperaciones
        })

df_metricas = pd.DataFrame(resumen_jugadores)

# Graficamos el Scatter Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_metricas, x='Precision', y='Recuperaciones', s=200, color='purple', edgecolors='black')

# Etiquetamos a cada jugador en el mapa de dispersión
for _, row in df_metricas.iterrows():
    plt.text(row['Precision'] + 0.5, row['Recuperaciones'] + 0.1, row['Jugador'], fontsize=10)

plt.title('Matriz de Rendimiento: Precisión vs Recuperación', fontsize=14, pad=15)
plt.xlabel('Precisión Estimada (%)')
plt.ylabel('Acciones de Recuperación / Anticipos')
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig('precision vs recuperacion_RECONSTRUIDO.png', bbox_inches='tight')
plt.show()